# Step 2: Expand HotpotQA Comparison Source Pool

目标：补第一批 `HotpotQA comparison` 候选题，支撑后续构造 `hp_comparison_set_01`。

这一版 notebook 对应：
- `source benchmark = HotpotQA`
- `target benchmark = 2WikiMultiHopQA`
- `N = 5`
- 当前 `HotpotQA comparison keep = 2`
- 第一批固定扩 `15` 个 raw candidates

输出：
- `candidate_batch_raw.csv`
- `candidate_batch_filtered.csv`
- `candidate_batch_for_taxonomy.csv`
- `candidate_batch_full.json`
- `candidate_batch_filtered_full.json`


In [1]:
!pip install -q datasets


In [2]:
import csv
import json
import random
import re
from collections import Counter
from pathlib import Path

from datasets import load_dataset

SEED = 42
RAW_BATCH_SIZE = 15
CURRENT_SPLIT = "validation"
OUTPUT_DIR = Path("02_hotpotqa_comparison_expansion")
OUTPUT_DIR.mkdir(exist_ok=True)

random.seed(SEED)

# 当前已经占用的 HotpotQA sampled tasks（来自 taxonomy.csv 的现有 10 题）
CURRENT_USED_HOTPOT_INDICES = [1547, 2054, 478, 3245, 1119, 3575, 978, 1668, 6149, 4237]

# 当前已经被标成 HotpotQA comparison keep 的两题
CURRENT_HOTPOTQA_COMPARISON_KEEP_INDICES = [478, 6149]

print(f"SEED = {SEED}")
print(f"RAW_BATCH_SIZE = {RAW_BATCH_SIZE}")
print(f"OUTPUT_DIR = {OUTPUT_DIR.resolve()}")


/Users/mac/miniforge3/envs/mind2web/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


SEED = 42
RAW_BATCH_SIZE = 15
OUTPUT_DIR = /Users/mac/studyspace/Knowledge-Markdown/capabilities/memory/2026_SelectTransfer/notebooks/02_hotpotqa_comparison_expansion


## 1. Load HotpotQA validation split

和 `01_sampling.ipynb` 保持一致，继续使用 `HotpotQA fullwiki / validation`。


In [3]:
hotpot = load_dataset("hotpot_qa", "fullwiki", split=CURRENT_SPLIT)
print(f"HotpotQA {CURRENT_SPLIT} size: {len(hotpot)}")
print(f"Columns: {hotpot.column_names}")
print("---")
print(hotpot[0])


Generating test split: 100%|██████████| 7405/7405 [00:00<00:00, 171459.29 examples/s]

HotpotQA validation size: 7405
Columns: ['id', 'question', 'answer', 'type', 'level', 'supporting_facts', 'context']
---
{'id': '5a8b57f25542995d1e6f1371', 'question': 'Were Scott Derrickson and Ed Wood of the same nationality?', 'answer': 'yes', 'type': 'comparison', 'level': 'hard', 'supporting_facts': {'title': ['Scott Derrickson', 'Ed Wood'], 'sent_id': [0, 0]}, 'context': {'title': ['Adam Collis', 'Ed Wood (film)', 'Tyler Bates', 'Doctor Strange (2016 film)', 'Hellraiser: Inferno', 'Sinister (film)', 'Deliver Us from Evil (2014 film)', 'Woodson, Arkansas', 'Conrad Brooks', 'The Exorcism of Emily Rose'], 'sentences': [['Adam Collis is an American filmmaker and actor.', ' He attended the Duke University from 1986 to 1990 and the University of California, Los Angeles from 2007 to 2010.', ' He also studied cinema at the University of Southern California from 1991 to 1997.', ' Collis first work was the assistant director for the Scott Derrickson\'s short "Love in the Ruins" (1995).', '

## 2. Recover current source-side state

把当前已经使用过的 HotpotQA 题和当前已有的 `comparison keep` 明确写出来，避免扩池时重复选入。


In [4]:
def task_id_from_index(idx: int) -> str:
    return f"hp_dev_{idx:04d}"

used_records = []
for idx in CURRENT_USED_HOTPOT_INDICES:
    item = hotpot[idx]
    used_records.append({
        "task_id": task_id_from_index(idx),
        "original_index": idx,
        "question": item["question"].strip(),
        "answer": item["answer"].strip(),
        "type": item.get("type", ""),
        "level": item.get("level", ""),
    })

comparison_keep_records = []
for idx in CURRENT_HOTPOTQA_COMPARISON_KEEP_INDICES:
    item = hotpot[idx]
    comparison_keep_records.append({
        "task_id": task_id_from_index(idx),
        "original_index": idx,
        "question": item["question"].strip(),
        "answer": item["answer"].strip(),
        "type": item.get("type", ""),
        "level": item.get("level", ""),
    })

print("Current used HotpotQA tasks:")
for r in used_records:
    print(f"- {r['task_id']} | type={r['type']} | {r['question']}")

print("\nCurrent HotpotQA comparison keep:")
for r in comparison_keep_records:
    print(f"- {r['task_id']} | {r['question']}")


Current used HotpotQA tasks:
- hp_dev_1547 | type=bridge | What was Iqbal F. Qadir on when he participated in an attack on a radar station located on western shore of the Okhamandal Peninsula?
- hp_dev_2054 | type=bridge | When did the park at which Tivolis Koncertsal is located open?
- hp_dev_0478 | type=comparison | What is the shared country of ancestry between Art Laboe and Scout Tufankjian?
- hp_dev_3245 | type=bridge | The school in which the Wilmslow Show is held is designated as what?
- hp_dev_1119 | type=bridge | Who will Billy Howle be seen opposite in the upcoming British drama film directed by Dominic Cooke?
- hp_dev_3575 | type=bridge | What animated movie, starring Danny Devito, featured music written and produced by Kool Kojak?
- hp_dev_0978 | type=bridge | Out of the actors who have played the role of Luc Deveraux in the Universal Soldier franchise, which actor has also starred in the movies Holby City, Doctor Strange, the Bourne Ultimatum and Zero Dark Thirty?
- hp_dev

## 3. Build comparison-only candidate pool

这一层只做预筛，不做最终 taxonomy judgment。

规则：
- 必须来自 HotpotQA 当前 split
- 必须 `type = comparison`
- 不能和当前已使用的 10 个 HotpotQA 题重复
- 题目和答案不能为空


In [5]:
def support_titles_from_item(item):
    sf = item.get("supporting_facts", {})
    if isinstance(sf, dict):
        return list(sf.get("title", []))
    return []

raw_comparison_pool = []
used_index_set = set(CURRENT_USED_HOTPOT_INDICES)

for idx in range(len(hotpot)):
    if idx in used_index_set:
        continue
    item = hotpot[idx]
    question = (item.get("question", "") or "").strip()
    answer = (item.get("answer", "") or "").strip()
    raw_type = item.get("type", "") or ""

    if raw_type != "comparison":
        continue
    if len(question) < 10 or len(answer) == 0:
        continue

    raw_comparison_pool.append({
        "task_id": task_id_from_index(idx),
        "dataset": "HotpotQA",
        "original_index": idx,
        "question": question,
        "answer": answer,
        "raw_type": raw_type,
        "level": item.get("level", ""),
        "support_titles": support_titles_from_item(item),
    })

print(f"Available comparison-only pool after excluding current used tasks: {len(raw_comparison_pool)}")
print("Example candidates:")
for c in raw_comparison_pool[:5]:
    print(f"- {c['task_id']} | {c['question']}")


Available comparison-only pool after excluding current used tasks: 1485
Example candidates:
- hp_dev_0000 | Were Scott Derrickson and Ed Wood of the same nationality?
- hp_dev_0003 | Are the Laleli Mosque and Esma Sultan Mansion located in the same neighborhood?
- hp_dev_0009 | Are Local H and For Against both from the United States?
- hp_dev_0013 | Are Random House Tower and 888 7th Avenue both used for real estate?
- hp_dev_0017 | Are Giuseppe Verdi and Ambroise Thomas both Opera composers ?


## 4. Sample the first raw batch

固定随机种子，从 comparison-only pool 中抽第一批 `15` 个 raw candidates。


In [6]:
raw_pool_shuffled = list(raw_comparison_pool)
random.shuffle(raw_pool_shuffled)
raw_batch = raw_pool_shuffled[:RAW_BATCH_SIZE]

print(f"Raw batch size: {len(raw_batch)}")
for i, c in enumerate(raw_batch, 1):
    print(f"[{i:02d}] {c['task_id']} | level={c['level']} | {c['question']}")


Raw batch size: 15
[01] hp_dev_4705 | level=hard | Of these two publication--Báiki and Sick--what type of publication is the one that was published most frequently?
[02] hp_dev_3635 | level=hard | Are Nikolaschka and White Russian made of beer?
[03] hp_dev_6333 | level=hard | Which genus of flowering plants has the most species, Caesalpinia or Achimenes?
[04] hp_dev_5118 | level=hard | Did Qionghai or Suining  have a population of 658,798 in 2002?
[05] hp_dev_1754 | level=hard | Were H. G. Wells and Rubén Darío both English?
[06] hp_dev_5052 | level=hard | Who was born first, Ana Kasparian or Andre Agassi?
[07] hp_dev_1691 | level=hard | Do the genuses Catopsis and Crataegus belong to the same family?
[08] hp_dev_6522 | level=hard | Does the genus Cunninghamia or Chilopsis contain only one species?
[09] hp_dev_5486 | level=hard | which is larger, Hunchun or Shijiazhuang?
[10] hp_dev_3041 | level=hard | Jim Sharman and Ava DuVernay both share what title concerning their actions leading 

## 5. Apply minimal filtering

这一层只做最小过滤：
- 去掉坏数据
- 去掉和现有 comparison keep 或批内样本几乎同模板的明显近重复
- 去掉支撑信息异常少的题

注意：这仍然不是最终 taxonomy judgment。


In [7]:
STOPWORDS = {
    "the", "a", "an", "of", "to", "in", "and", "or", "is", "was", "were", "did",
    "do", "does", "what", "which", "who", "when", "where", "how", "has", "have",
    "had", "with", "on", "for", "by", "from", "that", "this", "it", "its", "their",
    "his", "her", "as", "at", "between", "be", "been", "into", "than", "out", "shared"
}


def normalize_question(text: str) -> str:
    text = text.lower().strip()
    text = re.sub(r"\s+", " ", text)
    return text


def tokenize(text: str):
    toks = re.findall(r"[a-z0-9]+", text.lower())
    return {t for t in toks if t not in STOPWORDS}


def jaccard(a, b):
    if not a and not b:
        return 1.0
    if not a or not b:
        return 0.0
    return len(a & b) / len(a | b)

current_keep_token_sets = [tokenize(x["question"]) for x in comparison_keep_records]
filtered_batch = []
rejected_batch = []
selected_signatures = set()
selected_token_sets = []

for c in raw_batch:
    reasons = []
    qsig = normalize_question(c["question"])
    qtoks = tokenize(c["question"])

    if len(c["support_titles"]) < 2:
        reasons.append("support_titles_lt_2")

    if qsig in selected_signatures:
        reasons.append("duplicate_signature_in_batch")

    overlap_against_current = [jaccard(qtoks, toks) for toks in current_keep_token_sets]
    overlap_against_selected = [jaccard(qtoks, toks) for toks in selected_token_sets]
    all_overlaps = overlap_against_current + overlap_against_selected
    max_overlap = max(all_overlaps) if all_overlaps else 0.0

    if max_overlap >= 0.75:
        reasons.append("near_duplicate_question_pattern")

    if reasons:
        rejected_batch.append({
            **c,
            "max_question_overlap": round(max_overlap, 3),
            "reject_reasons": reasons,
        })
        continue

    filtered_batch.append({
        **c,
        "max_question_overlap": round(max_overlap, 3),
    })
    selected_signatures.add(qsig)
    selected_token_sets.append(qtoks)

print(f"Filtered batch size: {len(filtered_batch)}")
print(f"Rejected in minimal filtering: {len(rejected_batch)}")
print("--- kept ---")
for i, c in enumerate(filtered_batch, 1):
    print(f"[{i:02d}] {c['task_id']} | overlap={c['max_question_overlap']} | {c['question']}")

if rejected_batch:
    print("\n--- rejected ---")
    for c in rejected_batch:
        print(f"- {c['task_id']} | reasons={c['reject_reasons']} | {c['question']}")


Filtered batch size: 15
Rejected in minimal filtering: 0
--- kept ---
[01] hp_dev_4705 | overlap=0.0 | Of these two publication--Báiki and Sick--what type of publication is the one that was published most frequently?
[02] hp_dev_3635 | overlap=0.0 | Are Nikolaschka and White Russian made of beer?
[03] hp_dev_6333 | overlap=0.059 | Which genus of flowering plants has the most species, Caesalpinia or Achimenes?
[04] hp_dev_5118 | overlap=0.0 | Did Qionghai or Suining  have a population of 658,798 in 2002?
[05] hp_dev_1754 | overlap=0.0 | Were H. G. Wells and Rubén Darío both English?
[06] hp_dev_5052 | overlap=0.0 | Who was born first, Ana Kasparian or Andre Agassi?
[07] hp_dev_1691 | overlap=0.0 | Do the genuses Catopsis and Crataegus belong to the same family?
[08] hp_dev_6522 | overlap=0.167 | Does the genus Cunninghamia or Chilopsis contain only one species?
[09] hp_dev_5486 | overlap=0.0 | which is larger, Hunchun or Shijiazhuang?
[10] hp_dev_3041 | overlap=0.048 | Jim Sharman and A

## 6. Review filtered candidates

这一格是给人工快速扫一遍用的，确保这批题仍然像是合理的 comparison expansion batch。


In [8]:
for i, c in enumerate(filtered_batch, 1):
    print(f"[{i:02d}] {c['task_id']}")
    print(f"     Q: {c['question']}")
    print(f"     A: {c['answer']}")
    print(f"     support_titles: {c['support_titles']}")
    print(f"     overlap: {c['max_question_overlap']}")
    print()


[01] hp_dev_4705
     Q: Of these two publication--Báiki and Sick--what type of publication is the one that was published most frequently?
     A: satirical-humor magazine
     support_titles: ['Báiki', 'Sick (magazine)']
     overlap: 0.0

[02] hp_dev_3635
     Q: Are Nikolaschka and White Russian made of beer?
     A: no
     support_titles: ['Nikolaschka', 'Nikolaschka', 'White Russian (cocktail)', 'White Russian (cocktail)']
     overlap: 0.0

[03] hp_dev_6333
     Q: Which genus of flowering plants has the most species, Caesalpinia or Achimenes?
     A: Caesalpinia
     support_titles: ['Caesalpinia', 'Caesalpinia', 'Achimenes']
     overlap: 0.059

[04] hp_dev_5118
     Q: Did Qionghai or Suining  have a population of 658,798 in 2002?
     A: In 2002, Suining had a population of 658,798.
     support_titles: ['Qionghai', 'Qionghai', 'Suining', 'Suining']
     overlap: 0.0

[05] hp_dev_1754
     Q: Were H. G. Wells and Rubén Darío both English?
     A: no
     support_titles: ['H.

## 7. Export files

导出 5 份文件：
- raw batch CSV
- filtered batch CSV
- 可直接追加标注的 taxonomy CSV
- raw batch full JSON
- filtered batch full JSON


In [9]:
full_lookup = {c['task_id']: hotpot[c['original_index']] for c in raw_batch}

raw_csv_path = OUTPUT_DIR / "candidate_batch_raw.csv"
filtered_csv_path = OUTPUT_DIR / "candidate_batch_filtered.csv"
taxonomy_append_path = OUTPUT_DIR / "candidate_batch_for_taxonomy.csv"
full_json_path = OUTPUT_DIR / "candidate_batch_full.json"
filtered_full_json_path = OUTPUT_DIR / "candidate_batch_filtered_full.json"

with raw_csv_path.open("w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["task_id", "dataset", "original_index", "question", "answer", "raw_type", "level", "support_titles"])
    for c in raw_batch:
        writer.writerow([
            c["task_id"], c["dataset"], c["original_index"], c["question"], c["answer"],
            c["raw_type"], c["level"], "|".join(c["support_titles"])
        ])

with filtered_csv_path.open("w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["task_id", "dataset", "original_index", "question", "answer", "raw_type", "level", "support_titles", "max_question_overlap"])
    for c in filtered_batch:
        writer.writerow([
            c["task_id"], c["dataset"], c["original_index"], c["question"], c["answer"],
            c["raw_type"], c["level"], "|".join(c["support_titles"]), c["max_question_overlap"]
        ])

with taxonomy_append_path.open("w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["task_id", "dataset", "question", "answer", "reasoning_label", "keep_drop", "note"])
    for c in filtered_batch:
        writer.writerow([c["task_id"], c["dataset"], c["question"], c["answer"], "", "", ""])

raw_full = []
for c in raw_batch:
    raw_full.append({
        "task_id": c["task_id"],
        "dataset": c["dataset"],
        "question": c["question"],
        "answer": c["answer"],
        "raw": full_lookup[c["task_id"]],
    })

filtered_full = []
for c in filtered_batch:
    filtered_full.append({
        "task_id": c["task_id"],
        "dataset": c["dataset"],
        "question": c["question"],
        "answer": c["answer"],
        "max_question_overlap": c["max_question_overlap"],
        "raw": hotpot[c["original_index"]],
    })

with full_json_path.open("w", encoding="utf-8") as f:
    json.dump(raw_full, f, indent=2, ensure_ascii=False)

with filtered_full_json_path.open("w", encoding="utf-8") as f:
    json.dump(filtered_full, f, indent=2, ensure_ascii=False)

print("Saved files:")
for p in [raw_csv_path, filtered_csv_path, taxonomy_append_path, full_json_path, filtered_full_json_path]:
    print('-', p)


Saved files:
- 02_hotpotqa_comparison_expansion/candidate_batch_raw.csv
- 02_hotpotqa_comparison_expansion/candidate_batch_filtered.csv
- 02_hotpotqa_comparison_expansion/candidate_batch_for_taxonomy.csv
- 02_hotpotqa_comparison_expansion/candidate_batch_full.json
- 02_hotpotqa_comparison_expansion/candidate_batch_filtered_full.json


## 8. Quick summary for the next step

这里直接告诉你这批扩池之后要做什么。


In [10]:
existing_keep = len(CURRENT_HOTPOTQA_COMPARISON_KEEP_INDICES)
new_filtered = len(filtered_batch)
minimum_needed = max(0, 5 - existing_keep)
print(f"Existing HotpotQA comparison keep: {existing_keep}")
print(f"Filtered candidates ready for taxonomy annotation: {new_filtered}")
print(f"Minimum additional comparison keep still needed: {minimum_needed}")
print()
print("Next step:")
print("1. Read candidate_batch_filtered_full.json")
print("2. Annotate candidate_batch_for_taxonomy.csv using taxonomy_guideline.md")
print("3. Append annotated rows into pilot/taxonomy.csv")
print("4. Recount HotpotQA comparison keep")
print("5. If total >= 5, build hp_comparison_set_01")


Existing HotpotQA comparison keep: 2
Filtered candidates ready for taxonomy annotation: 15
Minimum additional comparison keep still needed: 3

Next step:
1. Read candidate_batch_filtered_full.json
2. Annotate candidate_batch_for_taxonomy.csv using taxonomy_guideline.md
3. Append annotated rows into pilot/taxonomy.csv
4. Recount HotpotQA comparison keep
5. If total >= 5, build hp_comparison_set_01


---

## Next steps

1. Download the exported folder `02_hotpotqa_comparison_expansion/`
2. Use `candidate_batch_filtered_full.json` to inspect supporting evidence
3. Fill `candidate_batch_for_taxonomy.csv`
4. Copy the annotated rows back into `pilot/taxonomy.csv`
5. Recount `HotpotQA comparison keep`
